# Feature Engineering

Feature engineering is the process of transforming existing campaign attributes into meaningful variables that can improve the performance of machine learning models.

The objective of this stage is to:

- Create meaningful campaign-level features
- Extract useful information from the campaign start date
- Transform categorical variables into machine-learning-compatible representations
- Capture campaign timing and lifecycle characteristics
- Identify and remove redundant or potentially leakage-prone variables
- Prepare a clean feature set for the predictive modeling stage

The engineered features will be used for predicting:

1. CTR
2. Conversion Rate
3. ROAS
4. Profit

In [1]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv(
    "../data/processed/campaign_data_cleaned.csv"
)

# Convert start_date to datetime
df["start_date"] = pd.to_datetime(df["start_date"])

print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)

Dataset Shape: (10000, 40)

Data Types:
campaign_id                                str
campaign_objective                         str
platform                                   str
ad_placement                               str
device_type                                str
operating_system                           str
creative_format                            str
creative_size                              str
ad_copy_length                             str
has_call_to_action                        bool
creative_emotion                           str
creative_age_days                        int64
target_audience_age                        str
target_audience_gender                     str
audience_interest_category                 str
income_bracket                             str
purchase_intent_score                      str
retargeting_flag                          bool
start_date                      datetime64[us]
quarter                                  int64
day_of_week         

In [2]:
# ============================================================
# TARGET VARIABLES
# ============================================================

target_variables = [
    "CTR",
    "conversion_rate",
    "ROAS",
    "profit"
]

print("Target Variables:")
for target in target_variables:
    print("-", target)

Target Variables:
- CTR
- conversion_rate
- ROAS
- profit


In [3]:
# ============================================================
# DATE-BASED FEATURES
# ============================================================

df["start_year"] = df["start_date"].dt.year
df["start_month"] = df["start_date"].dt.month
df["start_day"] = df["start_date"].dt.day
df["start_week"] = df["start_date"].dt.isocalendar().week.astype(int)

print(
    df[
        [
            "start_date",
            "start_year",
            "start_month",
            "start_day",
            "start_week"
        ]
    ].head()
)

  start_date  start_year  start_month  start_day  start_week
0 2024-03-06        2024            3          6          10
1 2024-01-26        2024            1         26           4
2 2025-05-15        2025            5         15          20
3 2024-07-21        2024            7         21          29
4 2025-03-09        2025            3          9          10


In [4]:
# ============================================================
# CAMPAIGN LIFECYCLE FEATURES
# ============================================================

df["creative_to_campaign_age_ratio"] = (
    df["creative_age_days"] /
    df["campaign_day"].replace(0, np.nan)
)

df["creative_to_campaign_age_ratio"] = (
    df["creative_to_campaign_age_ratio"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

display(
    df[
        [
            "creative_age_days",
            "campaign_day",
            "creative_to_campaign_age_ratio"
        ]
    ].head()
)

,creative_age_days,campaign_day,creative_to_campaign_age_ratio
0,72,34,2.117647
1,62,22,2.818182
2,79,32,2.468750
3,57,32,1.781250
4,17,52,0.326923


In [5]:
# ============================================================
# FEATURE ENGINEERING SUMMARY
# ============================================================

new_features = [
    "start_year",
    "start_month",
    "start_day",
    "start_week",
    "creative_to_campaign_age_ratio"
]

display(
    df[new_features].describe().round(2)
)

,start_year,start_month,start_day,start_week,creative_to_campaign_age_ratio
count,10000.00,10000.00,10000.00,10000.00,10000.00
mean,2024.56,6.21,15.57,25.08,2.52
std,0.58,3.56,8.75,15.48,6.44
min,2024.00,1.00,1.00,1.00,0.01
25%,2024.00,3.00,8.00,11.00,0.50
50%,2025.00,6.00,15.00,25.00,1.00
75%,2025.00,9.00,23.00,38.00,1.97
max,2026.00,12.00,31.00,52.00,90.00


In [8]:
# ============================================================
# 5.8 FEATURE SELECTION & LEAKAGE CHECK
# ============================================================

# Target variables
target_variables = [
    "CTR",
    "conversion_rate",
    "ROAS",
    "profit"
]

# Campaign attributes available before campaign performance
predictive_features = [
    "campaign_objective",
    "platform",
    "ad_placement",
    "device_type",
    "operating_system",
    "creative_format",
    "creative_size",
    "ad_copy_length",
    "has_call_to_action",
    "creative_emotion",
    "creative_age_days",
    "target_audience_age",
    "target_audience_gender",
    "audience_interest_category",
    "income_bracket",
    "purchase_intent_score",
    "retargeting_flag",
    "quarter",
    "day_of_week",
    "hour_of_day",
    "campaign_day",
    "quality_score",
    "industry_vertical",
    "budget_tier",
    "start_year",
    "start_month",
    "start_day",
    "start_week",
    "creative_to_campaign_age_ratio"
]

# Outcome-derived / post-performance variables
outcome_variables = [
    "impressions",
    "clicks",
    "conversions",
    "ad_spend",
    "revenue",
    "CPC",
    "CPA"
]

# Create feature inventory
feature_inventory = pd.DataFrame({
    "Category": (
        ["Predictive Feature"] * len(predictive_features)
        + ["Outcome / Potential Leakage"] * len(outcome_variables)
        + ["Target Variable"] * len(target_variables)
    ),
    
    "Variable": (
        predictive_features
        + outcome_variables
        + target_variables
    )
})

display(feature_inventory)

,Category,Variable
0,Predictive Feature,campaign_objective
1,Predictive Feature,platform
2,Predictive Feature,ad_placement
3,Predictive Feature,device_type
4,Predictive Feature,operating_system
5,Predictive Feature,creative_format
6,Predictive Feature,creative_size
7,Predictive Feature,ad_copy_length
8,Predictive Feature,has_call_to_action
9,Predictive Feature,creative_emotion


In [9]:
# ============================================================
# VERIFY FEATURE INVENTORY
# ============================================================

all_selected_variables = (
    predictive_features
    + outcome_variables
    + target_variables
)

missing_variables = [
    col for col in all_selected_variables
    if col not in df.columns
]

if missing_variables:
    print("Missing variables:", missing_variables)
else:
    print("All selected variables are present in the dataset.")

All selected variables are present in the dataset.


In [10]:
# ============================================================
# 5.8.1 REMOVE IDENTIFIER AND RAW DATE
# ============================================================

drop_columns = [
    "campaign_id",
    "start_date"
]

df_model = df.drop(columns=drop_columns)

print("Original Shape:", df.shape)
print("Modeling Dataset Shape:", df_model.shape)

Original Shape: (10000, 45)
Modeling Dataset Shape: (10000, 43)


In [11]:
# ============================================================
# 5.8.2 FINAL BASE PREDICTIVE FEATURES
# ============================================================

base_features = [
    "campaign_objective",
    "platform",
    "ad_placement",
    "device_type",
    "operating_system",
    "creative_format",
    "creative_size",
    "ad_copy_length",
    "has_call_to_action",
    "creative_emotion",
    "creative_age_days",
    "target_audience_age",
    "target_audience_gender",
    "audience_interest_category",
    "income_bracket",
    "purchase_intent_score",
    "retargeting_flag",
    "quarter",
    "day_of_week",
    "hour_of_day",
    "campaign_day",
    "quality_score",
    "industry_vertical",
    "budget_tier",
    "start_year",
    "start_month",
    "start_day",
    "start_week",
    "creative_to_campaign_age_ratio"
]

target_variables = [
    "CTR",
    "conversion_rate",
    "ROAS",
    "profit"
]

print("Number of base features:", len(base_features))
print("Number of targets:", len(target_variables))

Number of base features: 29
Number of targets: 4


In [12]:
# ============================================================
# 5.9 CATEGORICAL FEATURE CARDINALITY
# ============================================================

categorical_features = df[base_features].select_dtypes(
    include=["object", "string"]
).columns.tolist()

cardinality = (
    df[categorical_features]
    .nunique()
    .sort_values(ascending=False)
    .to_frame(name="Unique_Values")
)

display(cardinality)

,Unique_Values
day_of_week,7
audience_interest_category,6
platform,6
creative_size,6
ad_placement,6
creative_emotion,6
industry_vertical,6
target_audience_age,6
creative_format,6
campaign_objective,5


In [13]:
# ============================================================
# 5.10 FEATURE TYPE SEPARATION
# ============================================================

X_base = df[base_features]

categorical_features = X_base.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numerical_features = X_base.select_dtypes(
    include=["int64", "float64", "bool"]
).columns.tolist()

print("Categorical Features:", len(categorical_features))
print("Numerical Features:", len(numerical_features))

print("\nCategorical:")
print(categorical_features)

print("\nNumerical:")
print(numerical_features)

Categorical Features: 17
Numerical Features: 9

Categorical:
['campaign_objective', 'platform', 'ad_placement', 'device_type', 'operating_system', 'creative_format', 'creative_size', 'ad_copy_length', 'creative_emotion', 'target_audience_age', 'target_audience_gender', 'audience_interest_category', 'income_bracket', 'purchase_intent_score', 'day_of_week', 'industry_vertical', 'budget_tier']

Numerical:
['has_call_to_action', 'creative_age_days', 'retargeting_flag', 'quarter', 'hour_of_day', 'campaign_day', 'quality_score', 'start_week', 'creative_to_campaign_age_ratio']


In [14]:
# ============================================================
# 5.11 FEATURE ENCODING & PREPROCESSING PIPELINE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Separate boolean features
boolean_features = [
    "has_call_to_action",
    "retargeting_flag"
]

# Numerical features excluding boolean columns
numerical_features = [
    "creative_age_days",
    "quarter",
    "hour_of_day",
    "campaign_day",
    "quality_score",
    "start_week",
    "creative_to_campaign_age_ratio"
]

# Categorical features
categorical_features = [
    "campaign_objective",
    "platform",
    "ad_placement",
    "device_type",
    "operating_system",
    "creative_format",
    "creative_size",
    "ad_copy_length",
    "creative_emotion",
    "target_audience_age",
    "target_audience_gender",
    "audience_interest_category",
    "income_bracket",
    "purchase_intent_score",
    "day_of_week",
    "industry_vertical",
    "budget_tier"
]

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))
print("Boolean features:", len(boolean_features))

Categorical features: 17
Numerical features: 7
Boolean features: 2


In [15]:
# ============================================================
# PREPROCESSING TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "boolean",
            "passthrough",
            boolean_features
        )
    ],
    remainder="drop"
)

print("Preprocessing transformer created successfully.")

Preprocessing transformer created successfully.


In [16]:
# ============================================================
# 5.12 FINAL FEATURE DATASET PREPARATION
# ============================================================

# Feature matrix
X = df[base_features].copy()

# Target variables
y_ctr = df["CTR"].copy()
y_conversion_rate = df["conversion_rate"].copy()
y_roas = df["ROAS"].copy()
y_profit = df["profit"].copy()

print("Feature Matrix Shape:", X.shape)

print("\nTarget Shapes:")
print("CTR:", y_ctr.shape)
print("Conversion Rate:", y_conversion_rate.shape)
print("ROAS:", y_roas.shape)
print("Profit:", y_profit.shape)

Feature Matrix Shape: (10000, 29)

Target Shapes:
CTR: (10000,)
Conversion Rate: (10000,)
ROAS: (10000,)
Profit: (10000,)


In [21]:
# ============================================================
# MODELING READINESS CHECK
# ============================================================

print("=" * 55)
print("FINAL MODELING READINESS CHECK")
print("=" * 55)

print(f"Records: {len(X):,}")
print(f"Features: {X.shape[1]}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Boolean features: {len(boolean_features)}")

print("\nTargets:")
for target in target_variables:
    print(f"✓ {target}")

print("\nMissing feature values:", X.isna().sum().sum())

print("\nStatus: Feature dataset ready for modeling.")

FINAL MODELING READINESS CHECK
Records: 10,000
Features: 29
Categorical features: 17
Numerical features: 7
Boolean features: 2

Targets:
✓ CTR
✓ conversion_rate
✓ ROAS
✓ profit

Missing feature values: 0

Status: Feature dataset ready for modeling.


In [22]:
# ============================================================
# 5.14 SAVE FEATURE-ENGINEERED DATASET
# ============================================================

import os

# Create processed data directory if it does not exist
os.makedirs("../data/processed", exist_ok=True)

# Save feature-engineered dataset
feature_engineered_path = (
    "../data/processed/campaign_data_feature_engineered.csv"
)

df.to_csv(
    feature_engineered_path,
    index=False
)

print("Feature-engineered dataset saved successfully.")
print("Path:", feature_engineered_path)
print("Shape:", df.shape)

Feature-engineered dataset saved successfully.
Path: ../data/processed/campaign_data_feature_engineered.csv
Shape: (10000, 45)


In [23]:
# ============================================================
# 5.15 SAVE FEATURE CONFIGURATION
# ============================================================

import json

feature_config = {
    "base_features": base_features,

    "categorical_features": categorical_features,

    "numerical_features": numerical_features,

    "boolean_features": boolean_features,

    "target_variables": target_variables
}

config_path = "../data/processed/feature_config.json"

with open(config_path, "w") as file:
    json.dump(
        feature_config,
        file,
        indent=4
    )

print("Feature configuration saved successfully.")
print("Path:", config_path)

Feature configuration saved successfully.
Path: ../data/processed/feature_config.json


In [24]:
# ============================================================
# 5.16 FINAL SAVE VALIDATION
# ============================================================

import os

print("=" * 55)
print("FEATURE ENGINEERING SAVE VALIDATION")
print("=" * 55)

print(
    "Feature-engineered dataset:",
    "✓ Saved" if os.path.exists(feature_engineered_path)
    else "✗ Not Found"
)

print(
    "Feature configuration:",
    "✓ Saved" if os.path.exists(config_path)
    else "✗ Not Found"
)

FEATURE ENGINEERING SAVE VALIDATION
Feature-engineered dataset: ✓ Saved
Feature configuration: ✓ Saved


## 5.13 Feature Engineering Summary

Feature engineering was performed to transform the cleaned campaign dataset into a structured feature set suitable for machine learning.

### Key Steps Performed

1. **Created Date-Based Features**
   - Extracted `start_year`, `start_month`, `start_day`, and `start_week` from `start_date`.
   - These features capture potential seasonal and calendar-related patterns.

2. **Created Campaign Lifecycle Feature**
   - Created `creative_to_campaign_age_ratio` using `creative_age_days` and `campaign_day`.
   - This feature represents the age of the creative relative to the campaign duration.

3. **Removed Non-Predictive Identifiers**
   - `campaign_id` was excluded because it is only a unique identifier.
   - The original `start_date` was excluded from the modeling feature set after extracting its useful components.

4. **Identified Potential Data Leakage**
   - Variables such as `impressions`, `clicks`, `conversions`, `ad_spend`, `revenue`, `CPC`, and `CPA` were identified as outcome or post-performance variables.
   - These variables were excluded from the base predictive feature set to avoid data leakage.

5. **Defined Target Variables**
   - Four campaign performance metrics were defined as prediction targets:
     - `CTR`
     - `conversion_rate`
     - `ROAS`
     - `profit`

6. **Prepared Predictive Features**
   - A total of **29 base predictive features** were selected.
   - These consist of:
     - **17 categorical features**
     - **7 numerical features**
     - **2 boolean features**

7. **Prepared Preprocessing Pipeline**
   - Categorical features will be processed using One-Hot Encoding.
   - Numerical features will be standardized using StandardScaler.
   - Boolean features will be retained as binary values.
   - `handle_unknown="ignore"` will allow the model to handle unseen categorical values.

8. **Final Validation**
   - The final feature matrix contains **10,000 records and 29 predictive features**.
   - No missing values were found in the selected features or target variables.
   - The feature dataset is ready for model training.

### Final Feature Engineering Structure

| Component | Count |
|---|---:|
| Records | 10,000 |
| Predictive Features | 29 |
| Categorical Features | 17 |
| Numerical Features | 7 |
| Boolean Features | 2 |
| Target Variables | 4 |
| Missing Feature Values | 0 |

### Conclusion

The feature engineering stage transformed the cleaned campaign dataset into a consistent and leakage-aware feature set. The resulting features are ready for preprocessing and model development in the subsequent machine learning notebooks.

The prepared feature set will be used to develop separate predictive models for **CTR, Conversion Rate, ROAS, and Profit**.